In [4]:
import json
import pandas as pd
from pathlib import Path

# =========================
# 1. 경로 설정
# =========================

BASE_DIR = Path(".")
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
DATA_DIR = BASE_DIR / "data"
RESULT_DIR = BASE_DIR / "results"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 2. 변환 대상 정의
# =========================

targets = [
    {
        "checkpoint": CHECKPOINT_DIR / "checkpoint_sample_fnb_shorts_fail.json",
        "sample_csv": DATA_DIR / "sample_fnb_shorts_fail.csv",
        "output_csv": RESULT_DIR / "result_sample_fnb_shorts_fail.csv",
        "domain": "FnB",
        "success_label": "fail",
    },
    {
        "checkpoint": CHECKPOINT_DIR / "checkpoint_sample_fnb_shorts_success.json",
        "sample_csv": DATA_DIR / "sample_fnb_shorts_success.csv",
        "output_csv": RESULT_DIR / "result_sample_fnb_shorts_success.csv",
        "domain": "FnB",
        "success_label": "success",
    },
    {
        "checkpoint": CHECKPOINT_DIR / "checkpoint_sample_it_shorts_fail.json",
        "sample_csv": DATA_DIR / "sample_it_shorts_fail.csv",
        "output_csv": RESULT_DIR / "result_sample_it_shorts_fail.csv",
        "domain": "IT",
        "success_label": "fail",
    },
    {
        "checkpoint": CHECKPOINT_DIR / "checkpoint_sample_it_shorts_success.json",
        "sample_csv": DATA_DIR / "sample_it_shorts_success.csv",
        "output_csv": RESULT_DIR / "result_sample_it_shorts_success.csv",
        "domain": "IT",
        "success_label": "success",
    },
]

# =========================
# 3. checkpoint JSON → CSV 변환
# =========================

converted_dfs = []

for target in targets:
    checkpoint_path = target["checkpoint"]
    sample_csv_path = target["sample_csv"]
    output_csv_path = target["output_csv"]

    print("=" * 80)
    print(f"처리 중: {checkpoint_path.name}")

    # checkpoint JSON 로드
    if not checkpoint_path.exists():
        print(f"❌ 체크포인트 파일 없음: {checkpoint_path}")
        continue

    with open(checkpoint_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    result_df = pd.DataFrame(data)

    # domain, success_label 추가
    result_df["domain"] = target["domain"]
    result_df["success_label"] = target["success_label"]
    result_df["source_checkpoint"] = checkpoint_path.name

    # 원본 sample csv가 있으면 video_id 기준으로 일부 메타데이터 병합
    if sample_csv_path.exists():
        sample_df = pd.read_csv(sample_csv_path, encoding="utf-8-sig")

        # video_id 문자열 통일
        result_df["video_id"] = result_df["video_id"].astype(str).str.strip()
        sample_df["video_id"] = sample_df["video_id"].astype(str).str.strip()

        # 분석 결과에 없는 원본 컬럼만 붙이기
        meta_cols = [
            col for col in sample_df.columns
            if col not in result_df.columns and col != "video_id"
        ]

        # 너무 많은 컬럼을 붙이고 싶지 않으면 여기서 필요한 것만 선택해도 됨
        keep_cols = ["video_id"] + meta_cols

        result_df = result_df.merge(
            sample_df[keep_cols],
            on="video_id",
            how="left"
        )

    # CSV 저장
    result_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

    print(f"✅ 저장 완료: {output_csv_path}")
    print(f"행 수: {len(result_df)}")
    print(f"컬럼 수: {len(result_df.columns)}")

    converted_dfs.append(result_df)

# =========================
# 4. 4개 결과 통합 CSV 저장
# =========================

if converted_dfs:
    all_df = pd.concat(converted_dfs, ignore_index=True)

    all_output_path = RESULT_DIR / "result_sample_shorts_all_for_video_agent.csv"
    all_df.to_csv(all_output_path, index=False, encoding="utf-8-sig")

    print("=" * 80)
    print("✅ 전체 통합 CSV 저장 완료")
    print(f"저장 경로: {all_output_path}")
    print(f"전체 행 수: {len(all_df)}")

    display(all_df.head())
else:
    print("❌ 변환된 데이터가 없습니다.")

처리 중: checkpoint_sample_fnb_shorts_fail.json
✅ 저장 완료: results\result_sample_fnb_shorts_fail.csv
행 수: 52
컬럼 수: 75
처리 중: checkpoint_sample_fnb_shorts_success.json
✅ 저장 완료: results\result_sample_fnb_shorts_success.csv
행 수: 48
컬럼 수: 75
처리 중: checkpoint_sample_it_shorts_fail.json
✅ 저장 완료: results\result_sample_it_shorts_fail.csv
행 수: 58
컬럼 수: 75
처리 중: checkpoint_sample_it_shorts_success.json
✅ 저장 완료: results\result_sample_it_shorts_success.csv
행 수: 42
컬럼 수: 75
✅ 전체 통합 CSV 저장 완료
저장 경로: results\result_sample_shorts_all_for_video_agent.csv
전체 행 수: 200


,video_id,production_quality,lighting_style,color_mood,editing_pace,motion_graphic,video_format,first_3sec,background_style,top_colors,...,좋아요성과,댓글성과,조회수성과_상위1%,조회수성과_상위5%,좋아요성과_상위1%,좋아요성과_상위5%,댓글성과_상위1%,댓글성과_상위5%,score2,grade
0,VbWATcKlZ2Y,프로페셔널,인공조명,비비드,매우 빠름,핵심요소,애니메이션,인물,스튜디오,"[노란색, 흰색, 빨간색]",...,0.943130,0.409459,0.0,0.0,0.0,0.0,0.0,0.0,0.723787,실패
1,VxMVRz48OxU,프로페셔널,인공조명,중립,빠름,보조적,광고/CF,기업 로고,실내,"[회색, 녹색, 검은색]",...,0.828733,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.439959,실패
2,XXBuAxdGCf4,프로페셔널,인공조명,비비드,매우 빠름,보조적,광고/CF,인물,실내,"[빨간색, 흰색, 파란색]",...,0.618543,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.578244,실패
3,ddRC6B2nPpo,프로페셔널,인공조명,따뜻함,보통,핵심요소,광고/CF,텍스트,실내,"[빨간색, 검은색, 흰색]",...,0.700676,0.471514,0.0,0.0,0.0,0.0,0.0,0.0,0.525058,실패
4,VL2FgtMmI3g,프로페셔널,인공조명,따뜻함,보통,보조적,광고/CF,텍스트,실내,"[갈색, 금색, 흰색]",...,0.423689,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.455503,실패


In [5]:
import pandas as pd
from pathlib import Path

result_path = Path("results/result_sample_shorts_all_for_video_agent.csv")

df = pd.read_csv(result_path, encoding="utf-8-sig")

print(df.shape)
display(df.head())

display(
    df.groupby(["domain", "success_label"])
      .size()
      .reset_index(name="count")
)

print("중복 video_id 수:", df["video_id"].duplicated().sum())

ratio_cols = ["person_ratio", "face_ratio", "text_ratio"]

for col in ratio_cols:
    print(col, df[col].min(), df[col].max())

(200, 75)


,video_id,production_quality,lighting_style,color_mood,editing_pace,motion_graphic,video_format,first_3sec,background_style,top_colors,...,좋아요성과,댓글성과,조회수성과_상위1%,조회수성과_상위5%,좋아요성과_상위1%,좋아요성과_상위5%,댓글성과_상위1%,댓글성과_상위5%,score2,grade
0,VbWATcKlZ2Y,프로페셔널,인공조명,비비드,매우 빠름,핵심요소,애니메이션,인물,스튜디오,"['노란색', '흰색', '빨간색']",...,0.943130,0.409459,0.0,0.0,0.0,0.0,0.0,0.0,0.723787,실패
1,VxMVRz48OxU,프로페셔널,인공조명,중립,빠름,보조적,광고/CF,기업 로고,실내,"['회색', '녹색', '검은색']",...,0.828733,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.439959,실패
2,XXBuAxdGCf4,프로페셔널,인공조명,비비드,매우 빠름,보조적,광고/CF,인물,실내,"['빨간색', '흰색', '파란색']",...,0.618543,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.578244,실패
3,ddRC6B2nPpo,프로페셔널,인공조명,따뜻함,보통,핵심요소,광고/CF,텍스트,실내,"['빨간색', '검은색', '흰색']",...,0.700676,0.471514,0.0,0.0,0.0,0.0,0.0,0.0,0.525058,실패
4,VL2FgtMmI3g,프로페셔널,인공조명,따뜻함,보통,보조적,광고/CF,텍스트,실내,"['갈색', '금색', '흰색']",...,0.423689,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.455503,실패


,domain,success_label,count
0,FnB,fail,52
1,FnB,success,48
2,IT,fail,58
3,IT,success,42


중복 video_id 수: 0
person_ratio 0.0 1.0
face_ratio 0.0 1.0
text_ratio 0.0 1.0


# 1. 라이브러리 로드 + 통합 CSV 불러오기

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# 현재 노트북이 works/Hyeong_Uk/shorts_video_analysis 안에 있다고 가정
DATA_PATH = Path("results/result_sample_shorts_all_for_video_agent.csv")

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df.shape)
display(df.head())

데이터 크기: (200, 75)


,video_id,production_quality,lighting_style,color_mood,editing_pace,motion_graphic,video_format,first_3sec,background_style,top_colors,...,좋아요성과,댓글성과,조회수성과_상위1%,조회수성과_상위5%,좋아요성과_상위1%,좋아요성과_상위5%,댓글성과_상위1%,댓글성과_상위5%,score2,grade
0,VbWATcKlZ2Y,프로페셔널,인공조명,비비드,매우 빠름,핵심요소,애니메이션,인물,스튜디오,"['노란색', '흰색', '빨간색']",...,0.943130,0.409459,0.0,0.0,0.0,0.0,0.0,0.0,0.723787,실패
1,VxMVRz48OxU,프로페셔널,인공조명,중립,빠름,보조적,광고/CF,기업 로고,실내,"['회색', '녹색', '검은색']",...,0.828733,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.439959,실패
2,XXBuAxdGCf4,프로페셔널,인공조명,비비드,매우 빠름,보조적,광고/CF,인물,실내,"['빨간색', '흰색', '파란색']",...,0.618543,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.578244,실패
3,ddRC6B2nPpo,프로페셔널,인공조명,따뜻함,보통,핵심요소,광고/CF,텍스트,실내,"['빨간색', '검은색', '흰색']",...,0.700676,0.471514,0.0,0.0,0.0,0.0,0.0,0.0,0.525058,실패
4,VL2FgtMmI3g,프로페셔널,인공조명,따뜻함,보통,보조적,광고/CF,텍스트,실내,"['갈색', '금색', '흰색']",...,0.423689,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.455503,실패


# 2. 컬럼 확인

In [7]:
for i, col in enumerate(df.columns, start=1):
    print(i, col)

1 video_id
2 production_quality
3 lighting_style
4 color_mood
5 editing_pace
6 motion_graphic
7 video_format
8 first_3sec
9 background_style
10 top_colors
11 person_ratio
12 face_ratio
13 text_ratio
14 reason
15 avg_brightness
16 avg_blue
17 avg_green
18 avg_red
19 채널명
20 domain
21 success_label
22 source_checkpoint
23 title
24 channel_id
25 description
26 업로드일시
27 tags
28 조회수
29 좋아요수
30 댓글수
31 영상길이(초)
32 definition
33 license
34 embeddable
35 has_paid_product_placement
36 thumbnail
37 caption
38 final_url
39 instream_type
40 channel_handle
41 channel_tier
42 구독자수
43 description_missing_flag
44 tags_missing_flag
45 참여율(ER)
46 조회수 대비 좋아요율
47 조회수 대비 댓글률
48 wei
49 description_length
50 category_name
51 upload_year
52 upload_month
53 upload_dayofweek
54 upload_hour
55 tags_count
56 upload_quarter
57 upload_ym_quarter
58 upload_ymd
59 경과일수
60 도달률(RR)
61 일평균 조회수
62 RR_백분위
63 ER_백분위
64 score1
65 조회수성과
66 좋아요성과
67 댓글성과
68 조회수성과_상위1%
69 조회수성과_상위5%
70 좋아요성과_상위1%
71 좋아요성과_상위5%
72 댓글성과_상위1%
73 댓글성

# 3. 그룹별 샘플 수 확인

In [8]:
group_count = (
    df.groupby(["domain", "success_label"])
      .size()
      .reset_index(name="count")
)

display(group_count)

print("전체 데이터 수:", len(df))

,domain,success_label,count
0,FnB,fail,52
1,FnB,success,48
2,IT,fail,58
3,IT,success,42


전체 데이터 수: 200


# 4. 기본 품질 체크

In [10]:
# 4-1. 중복 video_id 확인
dup_count = df["video_id"].duplicated().sum()
print("중복 video_id 수:", dup_count)

if dup_count > 0:
    display(
        df[df["video_id"].duplicated(keep=False)]
        .sort_values("video_id")
    )

중복 video_id 수: 0


In [11]:
# 4-2. 결측치 확인
missing_summary = (
    df.isna()
      .sum()
      .reset_index()
)

missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_ratio"] = (missing_summary["missing_count"] / len(df)).round(3)

missing_summary = missing_summary.sort_values("missing_count", ascending=False)

display(missing_summary)

,column,missing_count,missing_ratio
66,댓글성과,15,0.075
33,embeddable,1,0.005
32,license,1,0.005
26,tags,1,0.005
36,caption,1,0.005
...,...,...,...
15,avg_blue,0,0.000
14,avg_brightness,0,0.000
11,face_ratio,0,0.000
13,reason,0,0.000


In [13]:
# 결측치가 있는 컬럼만 보고 싶으면:
display(missing_summary[missing_summary["missing_count"] > 0])

,column,missing_count,missing_ratio
66,댓글성과,15,0.075
33,embeddable,1,0.005
32,license,1,0.005
26,tags,1,0.005
36,caption,1,0.005
37,final_url,1,0.005
22,title,1,0.005
24,description,1,0.005
25,업로드일시,1,0.005
23,channel_id,1,0.005


In [14]:
# 4-3. 비율 컬럼 정상 범위 확인
ratio_cols = ["person_ratio", "face_ratio", "text_ratio"]

for col in ratio_cols:
    if col in df.columns:
        print(f"{col}")
        print("min:", df[col].min())
        print("max:", df[col].max())
        print("mean:", round(df[col].mean(), 3))
        print()

person_ratio
min: 0.0
max: 1.0
mean: 0.689

face_ratio
min: 0.0
max: 1.0
mean: 0.555

text_ratio
min: 0.0
max: 1.0
mean: 0.733



In [15]:
# 메타데이터가 안 붙은 것으로 의심되는 행 확인
meta_check_cols = ["title", "final_url", "업로드일시", "채널명", "조회수"]

missing_meta_rows = df[df["title"].isna() | df["final_url"].isna()]

print("메타데이터 매칭 실패 의심 행 수:", len(missing_meta_rows))
display(missing_meta_rows[["video_id", "domain", "success_label", "source_checkpoint"] + [col for col in meta_check_cols if col in df.columns]])

메타데이터 매칭 실패 의심 행 수: 1


,video_id,domain,success_label,source_checkpoint,title,final_url,업로드일시,채널명,조회수
92,pnz7WKQw58,FnB,success,checkpoint_sample_fnb_shorts_success.json,NaN,NaN,NaN,CU [씨유튜브],NaN


In [16]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")

sample_files = [
    "sample_fnb_shorts_fail.csv",
    "sample_fnb_shorts_success.csv",
    "sample_it_shorts_fail.csv",
    "sample_it_shorts_success.csv",
]

sample_all = []

for file in sample_files:
    temp = pd.read_csv(DATA_DIR / file, encoding="utf-8-sig")
    temp["source_sample_file"] = file
    temp["video_id"] = temp["video_id"].astype(str).str.strip()
    sample_all.append(temp)

sample_all = pd.concat(sample_all, ignore_index=True)

missing_ids = missing_meta_rows["video_id"].astype(str).str.strip().tolist()

display(sample_all[sample_all["video_id"].isin(missing_ids)])

,video_id,title,channel_id,채널명,description,업로드일시,tags,조회수,좋아요수,댓글수,...,조회수성과_상위1%,조회수성과_상위5%,좋아요성과_상위1%,좋아요성과_상위5%,댓글성과_상위1%,댓글성과_상위5%,score2,grade,success_label,source_sample_file


In [17]:
df["video_id"] = df["video_id"].astype(str).str.strip()
sample_all["video_id"] = sample_all["video_id"].astype(str).str.strip()

In [18]:
comment_perf_missing = df[df["댓글성과"].isna()]

print("댓글성과 결측 행 수:", len(comment_perf_missing))
display(comment_perf_missing[["video_id", "domain", "success_label", "채널명", "댓글수", "댓글성과"]])

댓글성과 결측 행 수: 15


,video_id,domain,success_label,채널명,댓글수,댓글성과
2,XXBuAxdGCf4,FnB,fail,셀렉스,0.0,NaN
20,vA2mq6we08Q,FnB,fail,셀렉스,0.0,NaN
53,HlMzYdlsZuA,FnB,success,삼립,1.0,NaN
56,wk6DRd6cdnU,FnB,success,그레인온,0.0,NaN
57,IRh4xqm5_dI,FnB,success,한맥 HANMAC,0.0,NaN
63,jsB9BBggAcI,FnB,success,Stella Artois Korea,0.0,NaN
64,carYemKUUHQ,FnB,success,다담몰,0.0,NaN
92,pnz7WKQw58,FnB,success,CU [씨유튜브],NaN,NaN
95,7tuSz8hg-iA,FnB,success,상쾌환 EASY TOMORROW,1.0,NaN
128,3gy8F7m_cKU,IT,fail,네이버클라우드 l NAVER Cloud,0.0,NaN


In [23]:
wrong_id = "pnz7WKQw58"

sample_all = []

for file in [
    "sample_fnb_shorts_fail.csv",
    "sample_fnb_shorts_success.csv",
    "sample_it_shorts_fail.csv",
    "sample_it_shorts_success.csv",
]:
    temp = pd.read_csv(f"data/{file}", encoding="utf-8-sig")
    temp["source_sample_file"] = file
    temp["video_id"] = temp["video_id"].astype(str).str.strip()
    sample_all.append(temp)

sample_all = pd.concat(sample_all, ignore_index=True)

candidate = sample_all[
    sample_all["video_id"].str.startswith(wrong_id, na=False)
]

display(candidate[["video_id", "채널명", "title", "final_url", "source_sample_file"]])

,video_id,채널명,title,final_url,source_sample_file
